In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, auc, roc_auc_score

Data Preparation
---
---

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# Check missing values
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical + ['converted']]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

categorical, numerical, df.isna().sum()

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'],
 lead_source                 0
 industry                    0
 employment_status           0
 location                    0
 annual_income               0
 number_of_courses_viewed    0
 interaction_count           0
 lead_score                  0
 converted                   0
 dtype: int64)

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)  # 3000, 1000, 1000

(3000, 1000, 1000)

Question 1: ROC AUC feature importance
---
---

In [6]:
for c in numerical:
    auc = roc_auc_score(y_train, df_train[c])
    auc_inv = roc_auc_score(y_train, -df_train[c])
    print(f'{c:26s} auc={auc:.4f}  inverted={auc_inv:.4f}  best={max(auc, auc_inv):.3f}')

annual_income              auc=0.6082  inverted=0.3918  best=0.608
number_of_courses_viewed   auc=0.7230  inverted=0.2770  best=0.723
interaction_count          auc=0.7649  inverted=0.2351  best=0.765
lead_score                 auc=0.7882  inverted=0.2118  best=0.788
